# 07 - Train Per-Crop Disease Classifier (EfficientNet-B2)

Stage 7: Train one disease classifier per crop (EfficientNet-B2).

Reads disease_dataset/<crop>/{train,val,test}/<disease>/ produced by
06_prepare_disease_dataset.py. Trains ONE crop at a time (pass --crop),
or all five sequentially if you leave --crop out -- useful since your
GPU will be busy a while and you may want to run crops one at a time
between other work.

Saves each crop's model + label list separately:
    models/disease_<crop>.pth
    models/disease_<crop>_labels.json

Set CROP_TO_TRAIN below and run -- set it to None to train all 5 crops
back-to-back in one run (only do this if your GPU has time to spare;
each crop takes as long as 02_train_crop_identifier.py did per epoch).

Install deps (same as 02_train_crop_identifier.py):
    pip install torch torchvision timm albumentations scikit-learn --break-system-packages

## Imports & Configuration

In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset
from torchvision.datasets import ImageFolder
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
from sklearn.metrics import classification_report
import cv2

DATA_ROOT = Path(r"Z:\Projects\Smart-Farming\Datasets\disease_dataset")
MODELS_DIR = Path(r"Z:\Projects\Smart-Farming\models")
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 25
LR = 3e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CROPS = ["Cotton", "Groundnut", "Pepper Bell", "Potato", "Tomato"]

train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.5),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(12, 24), hole_width_range=(12, 24), p=0.3),  # simulates leaf-spot occlusion
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

eval_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

## `AlbumentationsImageFolder`

In [ ]:
class AlbumentationsImageFolder(Dataset):
    def __init__(self, root, transform):
        self.base = ImageFolder(root)
        self.transform = transform
        self.classes = self.base.classes

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        path, label = self.base.samples[idx]
        try:
            image = cv2.imread(path)
            
            # Handle corrupt, unreadable, or missing image paths gracefully
            if image is None:
                raise ValueError("cv2.imread returned None")
                
        except Exception as e:
            print(f"\n[Warning] Error loading image at index {idx} ({path}): {e}. Using fallback sample.")
            fallback_path, _ = self.base.samples[0]
            image = cv2.imread(fallback_path)
            
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = self.transform(image=image)["image"]
        return image, label

## `build_weighted_sampler`

In [ ]:
def build_weighted_sampler(dataset):
    targets = [label for _, label in dataset.base.samples]
    class_counts = np.bincount(targets)
    class_weights = 1.0 / class_counts
    sample_weights = [class_weights[t] for t in targets]
    return WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

## `train_one_crop`

In [ ]:
from tqdm.auto import tqdm

def train_one_crop(crop_name):
    crop_slug = crop_name.replace(" ", "_")
    train_dir = DATA_ROOT / crop_slug / "train"
    val_dir = DATA_ROOT / crop_slug / "val"

    if not train_dir.exists():
        print(f"Skipping {crop_name}: {train_dir} not found (did 06 run for this crop?)")
        return

    print(f"\n{'=' * 20} Training disease classifier: {crop_name} {'=' * 20}")

    train_ds = AlbumentationsImageFolder(train_dir, train_tf)
    val_ds = AlbumentationsImageFolder(val_dir, eval_tf)
    num_classes = len(train_ds.classes)
    print(f"Classes ({num_classes}): {train_ds.classes}")

    if num_classes < 2:
        print(f"Skipping {crop_name}: fewer than 2 surviving classes after pruning in step 06.")
        return

    sampler = build_weighted_sampler(train_ds)
    
    # Set num_workers=0 to prevent Windows hanging/freezing issues
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = timm.create_model("efficientnet_b2", pretrained=True, num_classes=num_classes)
    model.to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    patience, patience_counter = 5, 0
    model_path = MODELS_DIR / f"disease_{crop_slug}.pth"
    labels_path = MODELS_DIR / f"disease_{crop_slug}_labels.json"
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        
        # 1. Training Loop with Live Progress Bar
        train_pbar = tqdm(train_loader, desc=f"  [{crop_name}] Epoch {epoch+1}/{EPOCHS} Train", leave=False)
        for images, labels in train_pbar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            train_pbar.set_postfix({"loss": f"{loss.item():.4f}"})
            
        scheduler.step()
        train_loss = running_loss / len(train_ds)

        # 2. Validation Loop with Live Progress Bar
        model.eval()
        correct, total = 0, 0
        all_preds, all_labels = [], []
        
        val_pbar = tqdm(val_loader, desc=f"  [{crop_name}] Epoch {epoch+1}/{EPOCHS} Val  ", leave=False)
        with torch.no_grad():
            for images, labels in val_pbar:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        val_acc = correct / total

        print(f"[{crop_name}] Epoch {epoch+1}/{EPOCHS} - train_loss: {train_loss:.4f} - val_acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), model_path)
            with open(labels_path, "w") as f:
                json.dump(train_ds.classes, f)
            print(f"  -> saved new best model (val_acc={val_acc:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"[{crop_name}] Early stopping triggered.")
                break

    print(f"[{crop_name}] Best val_acc: {best_val_acc:.4f}. Model saved to {model_path}")

    print(f"\n[{crop_name}] Val-set classification report (last epoch):")
    print(classification_report(all_labels, all_preds, target_names=train_ds.classes, digits=3, zero_division=0))

## Run

In [ ]:
CROP_TO_TRAIN = "Tomato"  # set to one of CROPS, or None to train all 5

crops_to_run = [CROP_TO_TRAIN] if CROP_TO_TRAIN else CROPS
for crop in crops_to_run:
    train_one_crop(crop)

In [ ]:
CROP_TO_TRAIN = "Cotton"  # set to one of CROPS, or None to train all 5

crops_to_run = [CROP_TO_TRAIN] if CROP_TO_TRAIN else CROPS
for crop in crops_to_run:
    train_one_crop(crop)

In [ ]:
CROP_TO_TRAIN = "Groundnut"  # set to one of CROPS, or None to train all 5

crops_to_run = [CROP_TO_TRAIN] if CROP_TO_TRAIN else CROPS
for crop in crops_to_run:
    train_one_crop(crop)

In [ ]:
CROP_TO_TRAIN = "Pepper Bell"  # set to one of CROPS, or None to train all 5

crops_to_run = [CROP_TO_TRAIN] if CROP_TO_TRAIN else CROPS
for crop in crops_to_run:
    train_one_crop(crop)

In [ ]:
CROP_TO_TRAIN = "Potato"  # set to one of CROPS, or None to train all 5

crops_to_run = [CROP_TO_TRAIN] if CROP_TO_TRAIN else CROPS
for crop in crops_to_run:
    train_one_crop(crop)